# Lab 5a — LoRA Fine-Tuning

In this notebook we take a small, general-purpose language model (Qwen2.5-0.5B) and teach it to answer science questions using **LoRA** (Low-Rank Adaptation). The idea is straightforward: instead of retraining the entire model (which would be expensive and slow), LoRA freezes the original weights and injects small trainable matrices into the model's attention layers. You end up training less than 1% of the total parameters.

By the end, we'll compare the model's answers before and after fine-tuning on the same set of questions.

## Background: Fine-Tuning and Why It Matters

Large language models like GPT-4, LLaMA, or Qwen are trained on massive amounts of text from the internet. This **pre-training** phase gives them broad language understanding — grammar, facts, reasoning patterns — but it doesn't make them good at any specific task. A pre-trained model is a generalist: it can write poetry, answer trivia, and generate code, but it does none of these particularly well out of the box.

**Fine-tuning** is the process of taking a pre-trained model and training it further on a smaller, task-specific dataset. The model's weights get updated to perform better on that particular task. For example:
- Fine-tune on customer support conversations → the model learns to answer support tickets
- Fine-tune on medical Q&A → the model learns medical terminology and diagnostic patterns
- Fine-tune on code review comments → the model learns to spot bugs and suggest fixes

There are two main approaches:

**Full fine-tuning** updates every parameter in the model. This gives maximum flexibility but is expensive — for a 7B parameter model, you'd need multiple GPUs and hours of training time. The entire model gets saved as a new copy.

**Parameter-efficient fine-tuning (PEFT)** only updates a small subset of parameters, keeping most of the model frozen. This is dramatically cheaper and often works just as well. LoRA, which we use in this lab, is the most popular PEFT method.

The key libraries involved:

| Library | Role |
|---------|------|
| **transformers** (HuggingFace) | Loads and runs the base model. This is the standard library for working with pre-trained models — it handles downloading weights, tokenization, and inference. |
| **peft** (HuggingFace) | Implements LoRA and other parameter-efficient methods. It wraps around a transformers model and injects the trainable adapters. |
| **trl** (HuggingFace) | Provides `SFTTrainer`, a training loop built specifically for supervised fine-tuning of language models. It handles data formatting, batching, and optimization. |
| **datasets** (HuggingFace) | Loads and preprocesses datasets. Handles downloading from the HuggingFace Hub, caching, and efficient data transformations. |
| **torch** (PyTorch) | The deep learning framework underneath everything. All model weights, computations, and gradient updates happen through PyTorch. |

## 1. Setup and imports

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import os

## 2. Load the SciQ dataset

SciQ is a science Q&A dataset from the Allen Institute for AI. Each sample has a question, a correct answer, and a support passage that explains the answer. We'll use the question-answer pairs for fine-tuning, and save the support passages for the RAG notebook later.

In [15]:
dataset = load_dataset("allenai/sciq", trust_remote_code=True)

print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples:  {len(dataset['test'])}")
print()

# Take a look at one example
sample = dataset['train'][0]
print("Question:", sample['question'])
print("Answer:  ", sample['correct_answer'])
print("Support: ", sample['support'][:200], "...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'allenai/sciq' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Train samples: 11679
Test samples:  1000

Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Answer:   mesophilic organisms
Support:  Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth tem ...


## 3. Load the base model

Qwen2.5-0.5B is a 500M parameter decoder-only transformer. The base model (not the Instruct variant) does text completion — given a prompt, it continues writing. It can answer science questions to some extent (it saw plenty of science text during pre-training), but its output tends to be noisy and unstructured. That gives us something to improve on with fine-tuning.

**Base model vs. Instruct model:** This is an important distinction. Qwen2.5-0.5B is a *base* model — its only job is **next-token prediction**. Given some text, it predicts what comes next. It was not trained to follow instructions or have a conversation.

An *Instruct* model (like Qwen2.5-0.5B-Instruct) goes through additional training phases after pre-training:
- **SFT (Supervised Fine-Tuning):** The model is shown examples of instructions paired with good responses, so it learns to follow commands.
- **RLHF (Reinforcement Learning from Human Feedback):** Human annotators rank multiple model outputs from best to worst for the same prompt. A separate, smaller Reward Model is then trained to "predict" these rankings and assign a score to any potential response. Finally, the language model is optimized (using reinforcement learning, specifically PPO — Proximal Policy Optimization) to produce outputs that this reward model scores highly. This multi-step alignment is what makes models like ChatGPT or Claude feel "helpful" and "safe" rather than just spitting out text completions.

We deliberately use the base model here. When you see messy outputs later — the model continuing a quiz, generating multiple-choice options nobody asked for, or repeating itself — that's not a bug. The model is doing exactly what it was trained to do: predict the most likely next tokens based on patterns in its training data, which included a lot of quiz-style web content. Fine-tuning will teach it a specific behavior instead.

### Under the Hood: Qwen2.5 Architecture

Before we move to fine-tuning, let's look at the "engine" we are modifying. These specifications determine how we configure our LoRA adapters in Section 5.

**1. Architecture Specifications (the 0.5B variant)**

| Component | Value | Why it matters |
|-----------|-------|----------------|
| **Transformer layers** | 24 | The model's depth. Each layer contains a Self-Attention module (the filter) and an FFN (the knowledge storage). |
| **Hidden dimension (d_model)** | 896 | The width of the vector representing each token. This number dictates the size of the weights we will be adapting. |
| **Attention heads** | 14 Query, 2 KV | Grouped-Query Attention (GQA). Instead of a 1:1 ratio, 7 Query heads share 1 Key-Value pair, reducing memory usage by 7×. |
| **Vocabulary** | 151,646 | Uses Byte-level BPE to represent any Unicode character, essential for complex scientific notation and formulas. |

**2. Grouped-Query Attention (GQA): The Memory Hack**

In older models (like GPT-2 or Llama 1), every "Query" head (the part that looks for patterns) had its own "Key" and "Value" head (the parts that store the context). This 1:1:1 ratio is called Multi-Head Attention (MHA).

**The Problem:** Storing a private "notebook" (KV pairs) for every "eye" (Query head) for 128,000 tokens would crash most GPUs.

**The Solution:** Qwen uses a 7:1 ratio. Each token has 14 Query heads to look at the world in 14 different ways, but they share only 2 sets of KV heads. This allows the model to maintain high "intelligence" while moving 7× less data through the GPU's memory.

**3. Key Innovations: Handling the "Long Tail"**

To process massive amounts of data without getting "confused," Qwen 2.5 uses two specific mathematical tricks:

- **Dual Chunk Attention (DCA):** To handle sequences up to 128,000 tokens, the model breaks text into chunks and uses a dual-pathway logic:
  - **Intra-Chunk (Fine-grained):** The model looks closely at words inside the same block to understand local logic.
  - **Inter-Chunk (Coarse-grained):** The model takes a "summary view" of words in different blocks.

  This "Dual" approach allows it to find a "needle in a haystack" (Inter-Chunk) and then read that specific section with perfect precision (Intra-Chunk).

- **RoPE with Adaptive Base Frequency (ABF):** Transformers use Rotary Position Embeddings (RoPE) to know the order of words. Think of this as a "clock hand" that rotates as you move through a sentence. ABF is a trick that slows down the rotation for very long sequences. This prevents the model from getting "dizzy" when it encounters context lengths (like 128K) that are much larger than what it saw in early training.

**4. Pre-Training & Scaling**

The model was exposed to **18 trillion tokens** in a two-phase pipeline:

- **Phase 1:** Initial training on 4,096-token sequences to learn core grammar and science facts.
- **Phase 2:** Progressive expansion to 128K tokens using a mixed-length strategy (75% long, 25% short) to ensure it stays sharp on both quick questions and long documents.

> **Note on MoE:** While our 0.5B model is "dense" (all 24 layers are always active), the larger Qwen2.5-Max uses Mixture-of-Experts (MoE) with 64 experts. It reduces compute costs by 30% by only activating the "experts" (sub-networks) relevant to the specific word being processed. This is how the Qwen family scales to massive sizes while remaining fast.


In [16]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

# Pick the best available device: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
).to(device)

# The tokenizer needs a pad token for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params / 1e6:.1f}M parameters")

Using device: mps


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded: 494.0M parameters


## 4. Test the vanilla model

Before any fine-tuning, let's see how the base model handles science questions. It may get some answers right — it's not a blank slate — but pay attention to the *format* of its output. You'll likely see correct-ish answers buried in quiz-style noise, self-evaluation text, or other junk the model picked up from its training data.

In [17]:
# These 10 questions come from the SciQ test set.
# We'll reuse them in every evaluation so the comparison is fair.
TEST_QUESTIONS = [
    "What type of organism is commonly used in preparation of foods such as bread and yogurt?",
    "What is the term for the property of a mineral that describes how it reflects light?",
    "What phenomenon makes global sea levels rise when ice sheets melt?",
    "What is the main gas found in the air we breathe?",
    "What is the name of the process by which plants make their own food using sunlight?",
    "What are the building blocks of proteins called?",
    "What type of rock is formed from cooled magma or lava?",
    "What force keeps planets in orbit around the sun?",
    "What is the smallest unit of life that can replicate independently?",
    "What part of the cell contains the genetic material?"
]

In [18]:
def generate_answer(model, tokenizer, question, max_new_tokens=80):
    """Generate an answer from the model given a question."""
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Only decode the generated part, not the prompt
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

### A note on tokenization

Language models don't read text the way we do. Before any text reaches the model, it gets broken into **tokens** — subword units that the model works with internally. The tokenizer converts text to token IDs (numbers) on the way in, and converts token IDs back to text on the way out.

Most modern models use **Byte-Pair Encoding (BPE)** or a variant of it. The idea: start with individual characters, then iteratively merge the most frequent pairs into single tokens. Common words like "the" become one token, while rare words get split into pieces (e.g., "photosynthesis" might become `["photos", "ynthesis"]`).

This matters for fine-tuning because:
- The model sees token IDs, not words. When we write a prompt like `Question: ...\nAnswer:`, the tokenizer splits it into a sequence of numbers. If we formatted the prompt differently (different punctuation, different spacing), it would produce different token IDs, and the model might not recognize the pattern it learned during training. Consistency between training and inference prompts is important.
- The EOS (end-of-sequence) token is just another token ID — the model learns to generate it when it should stop producing output
- Token count ≠ word count — a rough rule of thumb is 1 token ≈ 0.75 English words

Let's look at what the model actually receives as input. The prompt is just a plain string — we stick `Question:` in front and `Answer:` at the end. That's what gets tokenized and fed to the model. Nothing hidden, no system prompt, no special formatting.

In [6]:
# This is exactly what the model sees — nothing more, nothing less.
example_question = "What are the building blocks of proteins called?"
example_prompt = f"Question: {example_question}\nAnswer:"

print("--- Prompt sent to the model ---")
print(repr(example_prompt))
print()
print("--- How it looks to a human ---")
print(example_prompt)
print()

# And here's what the tokenizer turns it into
tokens = tokenizer(example_prompt, return_tensors="pt")
token_ids = tokens["input_ids"][0].tolist()
decoded_tokens = [tokenizer.decode([t]) for t in token_ids]

print(f"--- Tokenized ({len(token_ids)} tokens) ---")
for tid, tok in zip(token_ids, decoded_tokens):
    print(f"  {tid:>6d} -> {repr(tok)}")

--- Prompt sent to the model ---
'Question: What are the building blocks of proteins called?\nAnswer:'

--- How it looks to a human ---
Question: What are the building blocks of proteins called?
Answer:

--- Tokenized (13 tokens) ---
   14582 -> 'Question'
      25 -> ':'
    3555 -> ' What'
     525 -> ' are'
     279 -> ' the'
    4752 -> ' building'
   10010 -> ' blocks'
     315 -> ' of'
   27796 -> ' proteins'
    2598 -> ' called'
    5267 -> '?\n'
   16141 -> 'Answer'
      25 -> ':'


In [19]:
print("=" * 60)
print("VANILLA MODEL ANSWERS (before fine-tuning)")
print("=" * 60)

vanilla_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    answer = generate_answer(model, tokenizer, q)
    vanilla_answers.append(answer)
    print(f"\nQ{i}: {q}")
    print(f"A:  {answer}")

VANILLA MODEL ANSWERS (before fine-tuning)

Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
A:  The most common type of organism used in the production of bread and yogurt is the yeast.
 A single-select problem: Is the question answered in a satisfactory fashion?

Pick from:
 (1). yes
 (2). no
(1).

Q2: What is the term for the property of a mineral that describes how it reflects light?
A:  The term for the property of a mineral that describes how it reflects light is called refractive index.
 A single-select problem: Is the question answered in a satisfactory fashion?

Choose from:
 (1). yes
 (2). no
(1).

Q3: What phenomenon makes global sea levels rise when ice sheets melt?
A:  The melting of ice sheets causes sea levels to rise.
 A single-select problem: Is the question answered in a satisfactory fashion?

Select from the following.
 (1). yes
 (2). no
(1).

Q4: What is the main gas found in the air we breathe?
A:  The main gas found in t

The model often gets the core answer right (it's seen enough science text to know the basics), but it doesn't know when to stop. You'll notice quiz-style formatting, self-evaluation questions, and other noise after the actual answer. That's what fine-tuning will fix — teaching the model to give a clean answer and stop.

---

## 5. What is LoRA and why use it?

Full fine-tuning updates every weight in the model. For a 500M parameter model that's manageable, but for larger models (7B, 70B, etc.) it becomes impractical — you'd need massive amounts of GPU memory and time.

**LoRA** (Low-Rank Adaptation, [Hu et al. 2021](https://arxiv.org/abs/2106.09685)) takes a different approach. The core idea comes from an observation: when you fine-tune a large model, the weight updates tend to have low rank. That means the actual "useful" change to each weight matrix can be represented by a much smaller matrix.

Here's how it works:
- Freeze all original model weights — they don't change at all
- For selected weight matrices (typically in the attention layers), inject a pair of small matrices **A** and **B**
- The original weight W gets replaced by: `W' = W + B × A`
- Matrix A has shape `(d, r)` and B has shape `(r, d)`, where:
  - **d** is the **hidden dimension** of the model — the size of the internal vector that represents each token as it flows through the transformer layers. For Qwen2.5-0.5B, this is 896. Larger models have larger hidden dimensions (e.g., LLaMA-7B uses 4096). Every weight matrix in the attention layers has dimensions related to this number.
  - **r** is the **rank** — a small number you choose (we use 16). It controls how much capacity the adapter has. Think of it as the "bandwidth" for the update: rank 4 captures only broad patterns, rank 64 captures fine-grained ones.
- Only A and B are trained — the original W stays frozen

**The math behind the savings:**

Say the hidden dimension d = 1024 and we pick rank r = 16:
- A full weight update to one matrix: 1024 × 1024 = **1,048,576 parameters**
- The LoRA update: (1024 × 16) + (16 × 1024) = **32,768 parameters** — a 32× reduction
- Instead of updating the entire 1024×1024 matrix, we decompose the update into two thin rectangles (1024×16 and 16×1024) whose product approximates the full update

Across all targeted layers, you typically end up training less than 1% of the model's total parameters.

**Why target attention layers? What are Q, K, V, O?**

Transformer models process information through self-attention. At each layer, every token "looks at" every other token to decide what information to gather. This is done through four weight matrices:

- **Q (Query):** Transforms each token into a "question" vector — *"what am I looking for?"* For example, the token "it" in the sentence "The cat sat on the mat because **it** was tired" generates a query that essentially asks "who or what does 'it' refer to?"
- **K (Key):** Transforms each token into a "label" vector — *"what do I contain?"* The token "cat" generates a key that advertises "I'm an animal, a noun, the subject of the sentence."
- **V (Value):** Transforms each token into a "content" vector — *"what information should I pass along?"* Once attention decides that "it" should attend to "cat", the value vector of "cat" is what actually gets sent over.
- **O (Output projection):** Combines the gathered information from all attention heads back into a single vector per token.

**MultiHead(Q, K, V)** = Concat(head₁, ..., headₕ) × Wᴼ

The attention score between two tokens is the dot product of one token's Query with another's Key. High score = strong connection. The final output for each token is a weighted sum of the Value vectors, where the weights are these attention scores.

Research has shown that adapting **Q and V** gives the best results for most fine-tuning tasks. Q controls what the model looks for (adapting the "questions" it asks), and V controls what information gets passed forward (adapting the "answers" it collects). That's why we set `target_modules=["q_proj", "v_proj"]` below.

## 6. Prepare training data

We need to format each example as a prompt-completion pair. The model will learn that when it sees `Question: ... \nAnswer:`, it should produce the correct answer and stop.

**Why are we excluding the support passages?** Each SciQ sample comes with a `support` field — a paragraph that explains the answer. We deliberately leave it out of the training data. In this phase, we're training the model to answer from its own internal knowledge — a "closed-book" exam. The model has to learn to associate questions with answers using only what it has stored in its weights.

Later, in the RAG notebook, we'll flip this around. Those same support passages become the knowledge base that the model can reference at query time — turning it into an "open-book" exam. This split lets us clearly measure what the model learns on its own vs. what it gains from external retrieval.

In [8]:
def format_example(sample, tokenizer):
    """Turn a SciQ sample into a training string with an EOS token at the end.
    The EOS token teaches the model to stop generating after the answer."""
    text = (
        f"Question: {sample['question']}\n"
        f"Answer: {sample['correct_answer']}"
        f"{tokenizer.eos_token}"
    )
    return {"text": text}

# Format the training split
train_data = dataset["train"].map(lambda s: format_example(s, tokenizer))

# Sanity check — you should see the answer followed by <|endoftext|> or similar
print(train_data[0]["text"])
print()
print(f"EOS token: {repr(tokenizer.eos_token)}")
print(f"Total training examples: {len(train_data)}")

Question: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
Answer: mesophilic organisms<|endoftext|>

EOS token: '<|endoftext|>'
Total training examples: 11679


## 7. Configure LoRA

Here we set up the LoRA adapter. The main knobs:
- **r (rank):** Dimensionality of the low-rank matrices. 16 is a solid default — higher means more capacity but slower training. (higher values of rank can lead to overfitting to this specific dataset)
- **lora_alpha:** A scaling factor that controls how strongly the LoRA adapter's output affects the final result. The adapter output gets multiplied by `lora_alpha / r` before being added to the original weight. With r=16 and lora_alpha=32, the scaling factor is 32/16 = 2, meaning the adapter's contribution is amplified by 2×. Setting alpha higher than r gives the adapter more influence; setting it equal to r (alpha=16) gives a neutral 1× scaling. In practice, alpha = 2×r is a common starting point.
- **target_modules:** Which weight matrices get LoRA adapters. We target `q_proj` and `v_proj` (the Query and Value projections in each attention layer).
- **lora_dropout:** Dropout applied to the adapter during training. A small value (0.05) adds light regularization to prevent overfitting.

**How many parameters does this actually add?**

Because Qwen2.5 uses Grouped-Query Attention (GQA), the `v_proj` matrix is significantly smaller than the `q_proj` matrix (it only outputs 2 heads instead of 14). This means our LoRA adapters for the Value projections are smaller too.

- **q_proj params:** (896 × 16) + (16 × 896) = **28,672**
- **v_proj params:** (896 × 16) + (16 × 128) = **16,384**
- **Total:** (28,672 + 16,384) × 24 layers = **1,081,344 parameters**

This represents only **0.22%** of the model's total 494M parameters.

In [9]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

# Wrap the model with LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


That last line tells you exactly how many parameters are trainable vs frozen. You should see something like 0.5–1% of the total — that's the whole point of LoRA.

## 8. Training

We use `SFTTrainer` from the `trl` library, which handles the supervised fine-tuning loop for us. The configuration goes into `SFTConfig` — here are the key settings:

- **max_steps=250:** The total number of weight updates the trainer will perform. Each step processes one batch of data, computes the loss, and updates the adapter weights. We cap it at 250 rather than training for full epochs because it's enough to learn the Q&A pattern.
- **per_device_train_batch_size=8:** How many training examples are processed together in one step. Larger batches give a more accurate estimate of the gradient (the direction to update the weights), which leads to more stable training.
- **warmup_steps=50:** For the first 50 steps, the learning rate gradually increases from near-zero to its full value. This prevents large, destructive weight updates at the start when the model hasn't settled into a good direction yet.
- **lr_scheduler_type="cosine":** After warmup, the learning rate follows a cosine curve — it starts high and smoothly decreases toward zero. This lets the model make large adjustments early and fine-grained adjustments later.
- **max_length=128:** The maximum number of tokens per training example. Our Q&A pairs are short (typically 20-40 tokens), so 128 is plenty. Longer sequences use more memory and slow down training.
- **learning_rate=2e-4:** How big each weight update step is. Too high and the model overshoots; too low and it barely learns. 2e-4 (0.0002) is a well-tested default for LoRA fine-tuning.
- **dataset_text_field="text":** Tells the trainer which column in our dataset contains the formatted training strings (the `Question: ... Answer: ...<EOS>` strings we created earlier).
- **save_steps=125:** Save a checkpoint every 125 steps, so we have a mid-training snapshot in case something goes wrong.
- **report_to="none":** Disables logging to external services like Weights & Biases. We just watch the loss values in the notebook output.

In [10]:
sft_config = SFTConfig(
    output_dir="./models/training_checkpoints",
    max_steps=250,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=25,
    save_steps=125,
    fp16=False,                   # no half precision
    optim="adamw_torch",
    report_to="none",             # no wandb or similar
    max_length=128,
    dataset_text_field="text",    # which field in the dataset contains the training text
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_data,
    processing_class=tokenizer,
)

print("Starting training...")
print("Watch the loss values — they should decrease over time.")

Starting training...
Watch the loss values — they should decrease over time.


In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/stelioskliafas/Documents/GitHub/SemfeLabCourses/myvenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,2.663821
50,2.137614
75,2.098755
100,2.133026
125,2.062679
150,2.019776
175,2.065741
200,2.056925
225,2.021779
250,2.023265


TrainOutput(global_step=250, training_loss=2.128338165283203, metrics={'train_runtime': 273.2536, 'train_samples_per_second': 7.319, 'train_steps_per_second': 0.915, 'total_flos': 137606876909568.0, 'train_loss': 2.128338165283203})

## 9. Save the fine-tuned adapter

We save just the LoRA adapter weights — not the full model. The adapter is tiny (a few MB). In the RAG notebook we'll load the base model again and attach this adapter to it.

In [12]:
ADAPTER_PATH = "./models/sciq_lora_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

# Check what got saved
adapter_files = os.listdir(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")
print(f"Files: {adapter_files}")

# Show the adapter size
total_size = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f)) for f in adapter_files)
print(f"Total adapter size: {total_size / 1024:.1f} KB")

Adapter saved to ./models/sciq_lora_adapter
Files: ['adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'README.md', 'adapter_config.json', 'chat_template.jinja']
Total adapter size: 15399.6 KB


## 10. Test the fine-tuned model

Same questions, same generation function. Let's see what changed.

In [13]:
# Disable gradient checkpointing for inference (the trainer enables it during training)
model.gradient_checkpointing_disable()
model.eval()

print("=" * 60)
print("FINE-TUNED MODEL ANSWERS (after LoRA)")
print("=" * 60)

finetuned_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    answer = generate_answer(model, tokenizer, q)
    finetuned_answers.append(answer)
    print(f"\nQ{i}: {q}")
    print(f"A:  {answer}")

FINE-TUNED MODEL ANSWERS (after LoRA)

Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
A:  bacteria

Q2: What is the term for the property of a mineral that describes how it reflects light?
A:  optical property

Q3: What phenomenon makes global sea levels rise when ice sheets melt?
A:  global warming

Q4: What is the main gas found in the air we breathe?
A:  oxygen

Q5: What is the name of the process by which plants make their own food using sunlight?
A:  photosynthesis

Q6: What are the building blocks of proteins called?
A:  amino acids

Q7: What type of rock is formed from cooled magma or lava?
A:  sedimentary

Q8: What force keeps planets in orbit around the sun?
A:  gravity

Q9: What is the smallest unit of life that can replicate independently?
A:  cell

Q10: What part of the cell contains the genetic material?
A:  nucleus


## 11. Side-by-side comparison

Let's put the vanilla and fine-tuned answers next to each other so the difference is obvious.

In [14]:
EXPECTED_ANSWERS = [
    "Microorganisms (yeast, bacteria)",
    "Luster",
    "Thermal expansion of water",
    "Nitrogen",
    "Photosynthesis",
    "Amino acids",
    "Igneous rock",
    "Gravity",
    "Cell",
    "Nucleus"
]

for i in range(len(TEST_QUESTIONS)):
    print(f"{'='*70}")
    print(f"Q{i+1}: {TEST_QUESTIONS[i]}")
    print(f"  Expected:   {EXPECTED_ANSWERS[i]}")
    print(f"  Vanilla:    {vanilla_answers[i]}")
    print(f"  Fine-tuned: {finetuned_answers[i]}")
    print()

Q1: What type of organism is commonly used in preparation of foods such as bread and yogurt?
  Expected:   Microorganisms (yeast, bacteria)
  Vanilla:    The most common type of organism used in the production of bread and yogurt is the yeast.
 A single-select problem: Is the question answered in a satisfactory fashion?

Pick from:
 (1). yes
 (2). no
(1).
  Fine-tuned: bacteria

Q2: What is the term for the property of a mineral that describes how it reflects light?
  Expected:   Luster
  Vanilla:    The term for the property of a mineral that describes how it reflects light is called refractive index.
 A single-select problem: Is the question answered in a satisfactory fashion?

Choose from:
 (1). yes
 (2). no
(1).
  Fine-tuned: optical property

Q3: What phenomenon makes global sea levels rise when ice sheets melt?
  Expected:   Thermal expansion of water
  Vanilla:    The melting of ice sheets causes sea levels to rise.
 A single-select problem: Is the question answered in a satisfa

## Observations

Compare the two columns:
- The **vanilla model** often knows the answer but buries it in noise — quiz formatting, self-evaluation, repetition.
- The **fine-tuned model** gives clean, concise answers and stops. It learned the Q&A format and the stop signal (EOS token) from the training data.

Some answers will still be wrong. At 0.5B parameters, the model is small — it simply doesn't have the internal capacity to store a precise mapping for every scientific term and definition. It might say "oxygen" when the answer is "nitrogen," or "sedimentary" instead of "igneous." The knowledge is there in broad strokes, but not always at the level of specificity you'd need for a correct answer. This isn't a fine-tuning failure; it's a model size constraint.

That limitation is exactly what RAG addresses. In the next notebook, we'll give the model a search engine that can find the relevant passage at query time. The search engine doesn't need to "know" anything — it just needs to find the document that contains the answer. This is how RAG compensates for the limited capacity of smaller models.

---

**Next:** Open `lab5b_RAG_System.ipynb` to build the retrieval pipeline.